# Paper-Ready Metric Plots

PSNR / SSIM vs transmission budget. Error bars = 95% CI of the mean: `mean ± 1.96 × (σ/√n)`.

**Run from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

In [11]:
# ------------------------------- setting start ------------------------------ #
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
errorbar_color = "#3A3A3A"

# font
csfont = {'family': 'Times New Roman', 'serif': 'Times', 'size': 23}

# errorbar plot size
err_lw       = 1.5
err_capsize  = 4
err_capthick = 1.5

# figure size
figsize = (6.4, 4.8)

# set theme first, then rc — so seaborn doesn't clobber the font size
sns.set_theme(style="ticks", font="Times New Roman")
plt.rc('text', usetex=True)
plt.rc('font', **csfont)
# -------------------------------- setting end ------------------------------- #

In [12]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# Paths relative to dlapisgs-utility/ (set by the working-directory cell below)
SUMMARY_CSV = "output/0513_setup2_progressive/metrics/summary.csv"
OUT_DIR     = "plotting/paper"

# CSV column to group lines by
GROUP_BY = "weight_mode"

# Keys to exclude entirely (e.g. implementation mistakes)
EXCLUDE_KEYS = ["det_gamma_over_d2"]

# Per-key label / marker / color  (unmapped keys get auto fallback)
KEY_CONFIG = {
    "volume":         {"label": "Volume (view-indep.)",   "marker": "^", "color": color_palette[2]},
    "volume_over_d2": {"label": r"Vol/d$^2$ (view-dep.)", "marker": "D", "color": color_palette[3]},
    "screen_area":    {"label": "Screen Area",             "marker": "o", "color": color_palette[4]},
    "vd_lod":         {"label": "VD+LOD (baseline)",       "marker": "s", "color": color_palette[0]},
    "vd_lod_w":       {"label": "VD+LOD+W",                "marker": "^", "color": color_palette[1]},
    "vd_lod_c":       {"label": "VD+LOD+C",                "marker": "D", "color": color_palette[2]},
    "vd_lod_w_c":     {"label": "VD+LOD+W+C (proposed)",   "marker": "o", "color": color_palette[3]},
}

# Draw order (first entry plotted first, appears last in legend by default)
KEY_ORDER = {
    "weight_mode": ["screen_area", "volume_over_d2", "volume"],
    "scheme":      ["vd_lod", "vd_lod_w", "vd_lod_c", "vd_lod_w_c"],
}

DPI = 300

In [13]:
import os

# cd to dlapisgs-utility/ regardless of where nbconvert was invoked
_cwd = Path(os.getcwd())
_root = None
for _candidate in [_cwd, _cwd.parent]:
    if (_candidate / "utility_calculation.py").exists():
        _root = _candidate
        break
if _root is None:
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd
os.chdir(_root)
print(f"cwd: {Path.cwd()}")

cwd: /mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility


In [14]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()
print(f"Loaded {len(df)} rows | groups: {sorted(df[GROUP_BY].unique())} | budgets: {sorted(df['budget_mb'].unique())}")

Loaded 1050 rows | groups: ['screen_area', 'volume', 'volume_over_d2'] | budgets: [np.float64(20.0), np.float64(60.0), np.float64(100.0), np.float64(200.0), np.float64(500.0), np.float64(700.0), np.float64(1000.0)]


In [15]:
def aggregate_ci95(df, group_by):
    """mean ± 1.96·σ/√n per (group, budget, metric)"""
    records = []
    for key, gdf in df.groupby(group_by):
        for budget, bdf in gdf.groupby("budget_mb"):
            for metric in ("psnr", "ssim"):
                v = bdf[metric].dropna().values
                records.append({
                    group_by: key, "budget_mb": float(budget), "metric": metric,
                    "mean": v.mean(), "ci95": 1.96 * v.std(ddof=1) / np.sqrt(len(v)), "n": len(v),
                })
    return pd.DataFrame(records)

agg = aggregate_ci95(df, GROUP_BY)
print(agg.head(9).to_string(index=False))

weight_mode  budget_mb metric      mean     ci95  n
screen_area       20.0   psnr 12.067616 1.051026 50
screen_area       20.0   ssim  0.303152 0.063834 50
screen_area       60.0   psnr 13.869164 1.451623 50
screen_area       60.0   ssim  0.399806 0.078975 50
screen_area      100.0   psnr 15.228505 1.752886 50
screen_area      100.0   ssim  0.461441 0.086532 50
screen_area      200.0   psnr 18.281535 2.389487 50
screen_area      200.0   ssim  0.568526 0.101364 50
screen_area      500.0   psnr 26.493134 4.534499 50


In [16]:
def resolve_order(keys):
    preferred = KEY_ORDER.get(GROUP_BY, [])
    ordered   = [k for k in preferred if k in keys]
    return ordered + [k for k in sorted(keys) if k not in ordered]


def plot_metric(agg, metric, ylabel, out_stem, out_dir):
    sub  = agg[agg["metric"] == metric]
    keys = resolve_order(sub[GROUP_BY].unique().tolist())

    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=figsize)

    for i, key in enumerate(keys):
        cfg    = KEY_CONFIG.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  color_palette[i % len(color_palette)])

        kdf = sub[sub[GROUP_BY] == key].sort_values("budget_mb")
        ax.errorbar(
            kdf["budget_mb"].values, kdf["mean"].values, yerr=kdf["ci95"].values,
            marker=marker, color=color, linewidth=2, markersize=8,
            capsize=err_capsize, elinewidth=err_lw, capthick=err_capthick,
            label=label,
        )

    ax.set_xlabel(r"Budget (MiB)")
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper left", framealpha=0.9)

    # full box, ticks only on left and bottom
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [17]:
out_dir = Path(OUT_DIR)
plot_metric(agg, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/ssim_vs_budget.{png,eps}
Done.
